# Hello World Distributed with TorchTPU

This notebook demonstrates how to run distributed operations.
First, let's check the TPU topology.


In [10]:
from torch_tpu._internal.distributed import tpu_topology

try:
  topology, count = tpu_topology.get_tpu_topology()
  print(f"Detected TPU Topology: {topology}")
  print(f"Available TPU Cores (Count): {count}")
except Exception as e:
  print(f"Failed to detect topology: {e}")

Detected TPU Topology: 2,4,1
Available TPU Cores (Count): 8


## Define Worker Function

We define a function that will be executed on each distributed worker.
In this case, it performs an `all_reduce` operation.

```python
# Code snippet imported from hello_world_distributed.py
import torch
from torch import distributed as dist
from torch_tpu import api


def run_all_reduce():
  # Initialize tpu device.
  api.tpu_device()

  # Initialize the process group. The "tpu_dist" backend is used for distributed
  # TPU communication. Environment variables (RANK, WORLD_SIZE, etc.) are
  # typically set by the launcher.
  dist.init_process_group(backend="tpu_dist")
  ws = dist.get_world_size()
  x = torch.tensor([0.0, 1.0, ws], device="tpu")
  torch.distributed.all_reduce(x, op=dist.ReduceOp.SUM)

  expected = torch.tensor([0.0, ws, ws * ws])
  torch.testing.assert_close(x.cpu(), expected)
  print("result: ", x)
  print("✅ Distributed ran successfully!")


if __name__ == "__main__":
  run_all_reduce()

```


### Launch the runner

This approach uses `torchrun` combined with `singlehost_wrapper`.

`singlehost_wrapper` prepares the TPU environment (topology and addresses), and sets up standard distributed variables (`TORCH_TPU_TOPOLOGY`, `TORCH_TPU_SLICEBUILDER_ADDRESSES`, `WORLD_SIZE`, etc.).

In [ ]:
import os, subprocess, sys

# Populate TPU environment variables from singlehost_wrapper.
setup = subprocess.run(
    [
        sys.executable,
        "-m",
        "torch_tpu._internal.distributed.launchers.singlehost_wrapper",
    ],
    check=True,
    text=True,
    capture_output=True,
)
env = os.environ.copy()
for line in setup.stdout.splitlines():
  if "=" in line:
    k, _, v = line.partition("=")
    env[k.strip()] = v.strip().strip('"')

# Run distributed workers via torchrun.
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--nproc-per-node=8",
        "hello_world_distributed.py",
    ],
    env=env,
    text=True,
    capture_output=True,
)
result.check_returncode()
print(result.stdout)
print("✅ Done!")

Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
Device type: tpu, Device index: default
Initializing TPU distributed runtime
result:  result:  result: result:   result:  result:  result:  result:  tensor([ 0.,  8., 64.], device='tpu:0')
✅ Distributed ran successfully!
tensor([ 0.,  8., 64.], device='tpu:0')
✅ Distributed ran successfully!
tensor([ 0.,  8., 64.], device='tpu:0')
✅ Distributed ran successfully!
tensor([ 0.,  8., 64.], device='tpu:0')
✅ Distributed ran successfully!
tensor([ 0.,  8., 64.], 